# HW02 Data Science Academy
# Мурзабеков Эльдар 

## [Для всех] Чтение кода — без запуска
Для каждого фрагмента: **сначала** напишите в Markdown-ячейке ответ и одно предложение «почему», **потом** запустите и сравните. Ошибка в предсказании не снижает оценку — снижает отсутствие объяснения.

Мои предсказания и объяснения:
1. В созданный словарь r, перебирая через for c и используя безопасный метод get, считаем сколько раз была буква. Мой прогноз: A : 2, B : 1, 

In [1]:
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

{'A': 2, 'B': 1}


2. Создана функция f, которая добавляет 1 через append в конец. Изначально в переменной a хранится только 0 в списке. Использована функция 2 раза, будет 0, 1, 1

In [2]:
# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

[0, 1, 1]


3. В переменной s в словаре перебираем через for и нормализуем в строчный вид через lower. В конце использя print(len(s)) считаем количество, будет 2

In [3]:
# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

2


4. Через len берем длину и через for перебираем, но так-как использован {}, повторяющийся ключ перезаписывает своё значение, останется только ab и abc

In [4]:
# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

{'ab': 2, 'abc': 3}


5. Функция проверят, больше ли число нуля. print выведет 5 а не True, так-как в функции print а не return

In [5]:
# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)

5
None


In [6]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [7]:
import csv
with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "строк")

5463 строк


## [База]

In [8]:
from csv import DictReader

1. Напишите функции `to_num(s)` и `clean_city(s)` (6 городов, латиница и разный регистр → кириллица с заглавной).

In [9]:
def to_num(s):
    return float(s.replace(' ', '').replace(',', '.'))

In [10]:
def clean_city(s):
    s = s.strip().lower()
    kirill = {'almaty' : 'Алматы', 'astana' : 'Астана', 'shymkent' : 'Шымкент'}
    return kirill.get(s, s.capitalize()) 

In [11]:
print(clean_city('Almaty'))
print(clean_city('аСТана'))
print(clean_city('      Астана'))
print(clean_city('аЛмаТы       '))
print(clean_city('Astana'))
print(clean_city('shymkent'))

Алматы
Астана
Астана
Алматы
Астана
Шымкент


2. Сколько заказов в каждой категории (`category`)? Отсортируйте по убыванию.

In [12]:
cat = {}
for row in rows:
    if row['category'] in cat:
        cat[row['category']] += 1
    else:
        cat[row['category']] = 1
print(sorted(cat.items(), key=lambda x: x[1], reverse=True))

[('Одежда', 1529), ('Электроника', 1185), ('Дом', 1105), ('Красота', 991), ('Книги', 653)]


3. Для каждой категории посчитайте выручку оплаченных заказов и средний чек. Выведите таблицу f-строкой.

In [13]:
paid = {}
for p in rows:
    if p['status'] == 'оплачен' and p['category'] not in paid:
        paid[p['category']] = to_num(p['amount'])
    else:
        if p['status'] == 'оплачен':
            paid[p['category']] += to_num(p['amount'])

paid_count = 0
for row in rows:
    if row['status'] == 'оплачен':
        paid_count += 1

print(f'Выручка по категориям: {sum(paid.values())}')
print(f"Средний чек: {sum(paid.values()) / paid_count:.2f}")

Выручка по категориям: 101909360.0
Средний чек: 20780.86


4. Сколько заказов в каждом месяце? Месяц — это `order_date[:7]`.

In [14]:
order_months= {}
for month in rows:
    if month['order_date'][:7] in order_months:
        order_months[month['order_date'][:7]] += 1
    else:
        order_months[month['order_date'][:7]] = 1
print(order_months)

{'2026-09': 2782, '2026-10': 2681}


## [Стандарт]

5. Выручка оплаченных по месяцам и изменение октября к сентябрю в процентах.


In [15]:
revenue_months = {}
for month in rows:
    m = month['order_date'][:7]
    if month['status'] == 'оплачен' and m in revenue_months:
        revenue_months[m] += to_num(month['amount'])
    else:
        if month['status'] == 'оплачен':
            revenue_months[m] = to_num(month['amount']) 
print(f"Изменение октября к сентябрю в процентах: {(revenue_months['2026-10'] - revenue_months['2026-09']) / revenue_months['2026-09'] * 100:.1f}%")

Изменение октября к сентябрю в процентах: -22.7%


6. Доля отменённых заказов по каждому (чистому) городу, в процентах с одним знаком.

In [27]:
cancelled_cities = {}
total_cities = {}
for order in rows:
    if clean_city(order['city']) in total_cities:
        total_cities[clean_city(order['city'])] += 1
    else:
        total_cities[clean_city(order['city'])] = 1
    if order['status'] == 'отменён' and clean_city(order['city']) in cancelled_cities:
        cancelled_cities[clean_city(order['city'])] += 1
    else:
        if order['status'] == 'отменён':
            cancelled_cities[clean_city(order['city'])] = 1


for city, total in total_cities.items():
    cancelled = 0 
    if city in cancelled_cities:
        cancelled = cancelled_cities[city]
    percentage = cancelled / total * 100
    print(f"Доля отмененных заказов по городу {city}: {percentage:.1f}%")

Доля отмененных заказов по городу Алматы: 7.9%
Доля отмененных заказов по городу Астана: 8.8%
Доля отмененных заказов по городу Караганда: 8.9%
Доля отмененных заказов по городу Шымкент: 20.4%
Доля отмененных заказов по городу Атырау: 7.5%
Доля отмененных заказов по городу Актобе: 10.3%


7. Топ-5 клиентов (`customer_id`) по выручке оплаченных заказов. Кто на первом месте и что с ним не так? Ответьте в Markdown-ячейке.


In [32]:
customer_revenue = {}
for customer in rows:
    if customer['status'] == 'оплачен' and customer['customer_id'] in customer_revenue:
        customer_revenue[customer['customer_id']] += to_num(customer['amount'])
    else:
        if customer['status'] == 'оплачен':
            customer_revenue[customer['customer_id']] = to_num(customer['amount'])

print(f"Топ 5 клиентов по выручке оплаченных заказов {sorted(customer_revenue.items(), key=lambda x: x[1], reverse=True)[:5]}")

Топ 5 клиентов по выручке оплаченных заказов [('TEST', 29999970.0), ('C01627', 257180.0), ('C00485', 215920.0), ('C01338', 192080.0), ('C02122', 191670.0)]


На первом месте идет тестовый вариант, поэтому такой разрыв с остальными и искажение с общей выручкой

In [39]:
test_id = sorted(customer_revenue.items(), key=lambda x: x[1], reverse=True)[0][0]
print(test_id)

TEST


8. Сколько **уникальных** клиентов в каждом городе? Подсказка: словарь множеств.

In [44]:
city_customers = {}
for order in rows:
    if clean_city(order['city']) in city_customers:
        city_customers[clean_city(order['city'])].add(order['customer_id'])
    else:
        city_customers[clean_city(order['city'])] = set()
        city_customers[clean_city(order['city'])].add(order['customer_id'])

for city, customers in city_customers.items():
    print(f"Уникальные клиенты в городе {city}: {len(customers)}")


Уникальные клиенты в городе Алматы: 1351
Уникальные клиенты в городе Астана: 981
Уникальные клиенты в городе Караганда: 451
Уникальные клиенты в городе Шымкент: 655
Уникальные клиенты в городе Атырау: 138
Уникальные клиенты в городе Актобе: 358


## [Челлендж]

9. Найдите полные дубли строк: превратите каждую строку в `tuple(r.values())` и проверяйте через множество. Сколько их и в каком месяце?


In [49]:
seen = set()
duplicates = []
for order in rows:
    current = tuple(order.values())
    if current in seen:
        duplicates.append(current)
    else:
        seen.add(current)
print(f"Количество точных дубликатов: {len(duplicates)}")

dup_months = set()
for m in duplicates:
    dup_months.add(m[1][:7])
print(f"Месяцы: {dup_months}")

Количество точных дубликатов: 180
Месяцы: {'2026-09'}


10. Пересчитайте выручку по месяцам без дублей и без клиента из задачи 7. Как изменился процент из задачи 5?


In [56]:
unique_rows = []
seen = set()
for order in rows:
    current = tuple(order.values())
    if current in seen:
        pass 
    else:
        seen.add(current)
        unique_rows.append(order)
print(f"Уникальные без дублей: {len(unique_rows)}")


revenue = {}
for m in unique_rows:
    if m['status'] == 'оплачен' and m['customer_id'] != test_id:
        month = m['order_date'][:7]
        if month in revenue:
            revenue[month] += to_num(m['amount'])
        else:
            revenue[month] = to_num(m['amount'])

print(f"Процент без тестового искажения из задачи 7: {(revenue['2026-10'] - revenue['2026-09']) / revenue['2026-09'] * 100:.1f}")

Уникальные без дублей: 5283
Процент без тестового искажения из задачи 7: -2.3


Первоначальный расчёт показывал падение почти на 23%, но после очистки данных оно сократилось примерно до 2,3%